# Soup Kernel Challenge: run on a Kaggle T4

Kaggle settings (right panel): **Accelerator: GPU T4 x2**, **Internet: on** (needs a
phone-verified Kaggle account). Put your repository in the first code cell, then **Run All**.
About 10 minutes: base Soup and your solution are trained and measured, both adapters are scored,
and the rule checks run.

Private repository? Add a Kaggle secret named `GITHUB_TOKEN` (Add-ons → Secrets) with a
read-only GitHub token; the notebook uses it to clone and never prints or saves it. An optional
secret `HF_TOKEN` avoids Hugging Face rate limits. Do not run your own GPU work in this notebook while the measured runs
go: they need the GPUs idle.

Every Run All clones your repository afresh: push to the default branch and leave `COMMIT`
empty (or set it to the new commit). For the submission: make this notebook public, set
`COMMIT` to your final commit, save a version with Save & Run All and put its link in your
README and `SUBMISSION.md`. The repository must be open to the jury by the deadline.

In [ ]:
REPO_URL = "https://github.com/MakazhanAlpamys/soup-kernel-challenge.git"
COMMIT = ""

In [ ]:
import os, subprocess
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print(gpus)
assert "T4" in gpus, "Switch the accelerator to GPU T4 x2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
!python -m pip install -q "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest

In [ ]:
import base64, os, shutil, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("No GITHUB_TOKEN secret, cloning without it:", type(exc).__name__)
auth = []
if token:  # the token goes in a header for this one command: never in a URL, a file or the output
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)  # always a fresh copy of your code
clone = subprocess.run(["git", *auth, "clone", "-q", REPO_URL, "/kaggle/working/repo"],
                       capture_output=True, text=True)
if clone.returncode:
    raise SystemExit("git clone failed: " + clone.stderr)
if COMMIT:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

In [ ]:
!cd /kaggle/working/repo && python task/prepare_data.py
!cd /kaggle/working/repo && python -m pytest solution -q

In [ ]:
from pathlib import Path

solution = Path("/kaggle/working/repo/solution")
assert solution.exists(), "Папка solution не найдена!"

rmsnorm_code = r'''
import torch
import triton
import triton.language as tl


@triton.jit
def rms_fwd(X, W, Y, INV, H: tl.constexpr,
            EPS: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    col = tl.arange(0, BLOCK)
    x = tl.load(X + row * H + col, col < H, 0).to(tl.float32)
    w = tl.load(W + col, col < H, 0).to(tl.float32)
    mean_sq = tl.sum(x * x, 0) / H
    inv = tl.rsqrt(mean_sq + EPS)
    tl.store(Y + row * H + col, x * inv * w, col < H)
    tl.store(INV + row, inv)


@triton.jit
def rms_bwd(X, W, DY, INV, DX,
            H: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    col = tl.arange(0, BLOCK)
    x = tl.load(X + row * H + col, col < H, 0).to(tl.float32)
    w = tl.load(W + col, col < H, 0).to(tl.float32)
    dy = tl.load(DY + row * H + col, col < H, 0).to(tl.float32)
    inv = tl.load(INV + row)
    g = dy * w
    dot = tl.sum(g * x, 0) / H
    dx = inv * (g - x * inv * inv * dot)
    tl.store(DX + row * H + col, dx, col < H)


class RMSNormFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, weight, eps):
        shape = x.shape
        h = shape[-1]
        x2 = x.contiguous().view(-1, h)
        w = weight.contiguous()
        y = torch.empty_like(x2)
        inv = torch.empty((x2.shape[0],), device=x.device, dtype=torch.float32)
        block = triton.next_power_of_2(h)

        rms_fwd[(x2.shape[0],)](
            x2, w, y, inv, h, eps, block
        )

        ctx.save_for_backward(x2, w, inv)
        ctx.shape = shape
        ctx.block = block
        return y.view(shape)

    @staticmethod
    def backward(ctx, grad_output):
        x, weight, inv = ctx.saved_tensors
        dy = grad_output.contiguous().view_as(x)
        dx = torch.empty_like(x)

        rms_bwd[(x.shape[0],)](
            x, weight, dy, inv, dx, x.shape[1], ctx.block
        )

        dw = (
            dy.float() * x.float() * inv[:, None]
        ).sum(dim=0).to(weight.dtype)

        return dx.view(ctx.shape), dw, None


def patch_rmsnorm():
    from transformers.models.qwen2 import modeling_qwen2

    cls = modeling_qwen2.Qwen2RMSNorm

    def forward(self, hidden_states):
        if not hidden_states.is_cuda:
            x = hidden_states.float()
            variance = x.pow(2).mean(-1, keepdim=True)
            return self.weight * (
                x * torch.rsqrt(variance + self.variance_epsilon)
            ).to(hidden_states.dtype)

        return RMSNormFunction.apply(
            hidden_states, self.weight, self.variance_epsilon
        )

    cls.forward = forward
'''

(solution / "rmsnorm.py").write_text(rmsnorm_code)

init_code = '''from .swiglu import patch_qwen2_mlp
from .rmsnorm import patch_rmsnorm

def apply():
    patch_qwen2_mlp()
    patch_rmsnorm()
'''

(solution / "_init_.py").write_text(init_code)

print("RMSNorm создан и подключён!")
print("Файлы:", [f.name for f in solution.iterdir()])

In [ ]:
%cd /kaggle/working/repo
!python -m pytest solution -q

In [ ]:
!cd /kaggle/working/repo && python harness/run_task.py

In [ ]:
!ls -la /kaggle/working/repo

In [ ]:
!cd /kaggle/working/repo && git remote -v

In [ ]:
!cd /kaggle/working/repo && git remote set-url origin https://github.com/shinzo-imperor/soup-kernel-challenge.git

In [ ]:
!cd /kaggle/working/repo && git remote -v

In [ ]:
!ls -lah /kaggle/working/repo/runs

In [ ]:
import json

path = "/kaggle/working/repo/runs/result.json"

with open(path) as f:
    data = json.load(f)

print(json.dumps(data, indent=2, ensure_ascii=False)[:12000])

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/repo/solution")

for name in ["_init_.py", "swiglu.py"]:
    f = p / name
    print(f"\n{'='*20} {name} {'='*20}")
    print(f.read_text()[:7000])

In [ ]:
from pathlib import Path

repo = Path("/kaggle/working/repo")

print("Папка repo существует:", repo.exists())
print("Папки solution найдены:")

for p in Path("/kaggle/working").rglob("solution"):
    if p.is_dir() and (p / "swiglu.py").exists():
        print(p)
        for name in ["_init_.py", "swiglu.py"]:
            f = p / name
            print(f"\n===== {name} =====")
            if f.exists():
                print(f.read_text()[:7000])
            else:
                print("Файл не найден")

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/repo/solution")

print("Все файлы в solution:")
for f in sorted(p.iterdir()):
    print(repr(f.name))

print("\nСодержимое всех Python-файлов:")
for f in sorted(p.glob("*.py")):
    print(f"\n{'='*15} {f.name} {'='*15}")
    print(f.read_text()[:10000])

In [ ]:
import inspect
from transformers.models.qwen2 import modeling_qwen2

print(inspect.getsource(modeling_qwen2.Qwen2RMSNorm))

In [ ]:
from pathlib import Path

repo = Path("/kaggle/working/repo")
solution = repo / "solution"

rmsnorm_code = r'''
import torch
import triton
import triton.language as tl


@triton.jit
def rms_fwd(X, W, Y, INV, N: tl.constexpr, H: tl.constexpr,
            EPS: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    col = tl.arange(0, BLOCK)
    x = tl.load(X + row * H + col, col < H, 0).to(tl.float32)
    w = tl.load(W + col, col < H, 0).to(tl.float32)
    mean_sq = tl.sum(x * x, 0) / H
    inv = tl.rsqrt(mean_sq + EPS)
    y = x * inv * w
    tl.store(Y + row * H + col, y, col < H)
    tl.store(INV + row, inv)


@triton.jit
def rms_bwd(X, W, DY, INV, DX,
            H: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    col = tl.arange(0, BLOCK)
    x = tl.load(X + row * H + col, col < H, 0).to(tl.float32)
    w = tl.load(W + col, col < H, 0).to(tl.float32)
    dy = tl.load(DY + row * H + col, col < H, 0).to(tl.float32)
    inv = tl.load(INV + row)
    weighted_grad = dy * w
    dot = tl.sum(weighted_grad * x, 0) / H
    dx = inv * (weighted_grad - x * inv * inv * dot)
    tl.store(DX + row * H + col, dx, col < H)


class RMSNormFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, weight, eps):
        original_shape = x.shape
        h = original_shape[-1]
        x2 = x.contiguous().view(-1, h)
        w = weight.contiguous()
        y = torch.empty_like(x2)
        inv = torch.empty((x2.shape[0],), device=x.device, dtype=torch.float32)
        block = triton.next_power_of_2(h)

        rms_fwd[(x2.shape[0],)](
            x2, w, y, inv, x2.shape[0], h, eps, block
        )

        ctx.save_for_backward(x2, w, inv)
        ctx.original_shape = original_shape
        ctx.block = block
        return y.view(original_shape)

    @staticmethod
    def backward(ctx, grad_output):
        x, weight, inv = ctx.saved_tensors
        dy = grad_output.contiguous().view_as(x)
        dx = torch.empty_like(x)

        rms_bwd[(x.shape[0],)](
            x, weight, dy, inv, dx, x.shape[1], ctx.block
        )

        # Gradient for the scale parameter.
        h = x.shape[1]
        x_float = x.float()
        dy_float = dy.float()
        inv_col = inv[:, None]
        dw = (dy_float * x_float * inv_col).sum(dim=0).to(weight.dtype)

        return dx.view(ctx.original_shape), dw, None


def patch_rmsnorm():
    from transformers.models.qwen2 import modeling_qwen2

    cls = modeling_qwen2.Qwen2RMSNorm

    def forward(self, hidden_states):
        if not hidden_states.is_cuda:
            x = hidden_states.float()
            variance = x.pow(2).mean(-1, keepdim=True)
            return self.weight * (x * torch.rsqrt(
                variance + self.variance_epsilon
            )).to(hidden_states.dtype)

        return RMSNormFunction.apply(
            hidden_states, self.weight, self.variance_epsilon
        )

    cls.forward = forward
'''

(solution / "rmsnorm.py").write_text(rmsnorm_code)

init_code = '''from .swiglu import patch_qwen2_mlp
from .rmsnorm import patch_rmsnorm


def apply():
    patch_qwen2_mlp()
    patch_rmsnorm()
'''

(solution / "_init_.py").write_text(init_code)

print("Создан solution/rmsnorm.py")
print("Обновлён solution/_init_.py")

In [ ]:
%cd /kaggle/working/repo
!python -m pytest solution -q

In [ ]:
import os
from pathlib import Path

os.chdir("/kaggle/working")

print("Текущая папка:", os.getcwd())
print("Содержимое /kaggle/working:")

for p in Path("/kaggle/working").iterdir():
    print(p.name)

In [ ]:
import os
from pathlib import Path

os.chdir("/kaggle/working")

repo = Path("/kaggle/working/repo")

print("Repo существует:", repo.exists())

if repo.exists():
    print("RMSNorm существует:", (repo / "solution/rmsnorm.py").exists())
    print("Init существует:", (repo / "solution/_init_.py").exists())
else:
    print("Содержимое working:")
    print(list(Path("/kaggle/working").iterdir()))

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/repo/solution")

print("Папка существует:", p.exists())

if p.exists():
    for f in sorted(p.iterdir()):
        print(f.name)

In [ ]:
import json
from pathlib import Path

files = list(Path("/kaggle/working").glob("*.ipynb"))
print("Ноутбуки:", [f.name for f in files])

for nb in files:
    data = json.loads(nb.read_text())
    print("\nНОУТБУК:", nb.name)
    for i, cell in enumerate(data["cells"]):
        source = "".join(cell.get("source", []))
        if "git clone" in source or "REPO_URL" in source or "soup train" in source:
            print(f"\n--- Ячейка {i} ---")
            print(source[:1500])

In [ ]:
!cd /kaggle/working/repo && python harness/check_rules.py
!cd /kaggle/working/repo && python harness/score.py runs/result.json

In [ ]:
import json
from pathlib import Path

p = Path("/kaggle/working/repo/runs/result.json")
data = json.loads(p.read_text())

print(json.dumps(data.get("solution", {}), indent=2))

In [ ]:
import json
from pathlib import Path

p = Path("/kaggle/working/repo/runs/result.json")

print("Файл существует:", p.exists())

if p.exists():
    data = json.loads(p.read_text())
    print(json.dumps(data, indent=2)[:15000])

In [ ]:
import json
from pathlib import Path

d = json.loads(Path(
    "/kaggle/working/repo/runs/result.json"
).read_text())

m = d["runs"]["solution"][0]["metrics"]

for key in ["solution", "apply_s", "own_kernel_ms",
            "triton_launches", "triton_files",
            "triton_watch_error", "steps"]:
    print(key, "=", m.get(key))

In [ ]:
%cd /kaggle/working/repo
!python harness/run_task.py --only solution --profile

In [ ]:
%cd /kaggle/working/repo

!grep -n -E "RMSNorm|rmsnorm|patch_rmsnorm|patch_qwen2_mlp" solution/_init_.py solution/rmsnorm.py

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/repo/solution/_init_.py")

p.write_text("""from .swiglu import patch_qwen2_mlp
from .rmsnorm import patch_rmsnorm

def apply():
    patch_qwen2_mlp()
    patch_rmsnorm()
""")

print("Правильный _init_.py создан:")
print(p.read_text())
print("Файл существует:", p.exists())

In [ ]:
%cd /kaggle/working/repo

!python -c "import solution; print('Импортирован файл:', solution.__file__); print('Функция RMSNorm:', solution.patch_rmsnorm)"

In [ ]:
from pathlib import Path

folder = Path("/kaggle/working/repo/solution")

print("Файлы:")
for f in folder.iterdir():
    print(f.name)

print("\nСодержимое _init_.py:")
print((folder / "_init_.py").read_text())

print("\nЕсть ли patch_rmsnorm в rmsnorm.py:")
print("patch_rmsnorm" in (folder / "rmsnorm.py").read_text())

In [ ]:
from pathlib import Path

init_file = Path("/kaggle/working/repo/solution/_init_.py")

init_file.write_text(
    "from .swiglu import patch_qwen2_mlp\n"
    "from .rmsnorm import patch_rmsnorm\n\n"
    "def apply():\n"
    "    patch_qwen2_mlp()\n"
    "    patch_rmsnorm()\n"
)

print("ПРАВИЛЬНЫЙ ФАЙЛ:")
print(init_file)
print(init_file.read_text())

In [ ]:
%cd /kaggle/working/repo
!python -c "import solution; print(solution.__file__); print(solution.patch_rmsnorm)"

In [ ]:
%cd /kaggle/working/repo

!echo "=== _init_.py ==="
!cat solution/_init_.py

!echo "=== rmsnorm.py ==="
!head -n 15 solution/rmsnorm.py

!echo "=== IMPORT CHECK ==="
!python -c "import solution; print('FILE:', solution.__file__); print('DIR:', [x for x in dir(solution) if 'rms' in x.lower() or x == 'apply'])"

In [ ]:
from pathlib import Path

folder = Path("/kaggle/working/repo/solution")

for name in ["_init.py", "_init.py"]:
    p = folder / name
    print(f"\n===== {name} =====")
    print(p.read_text() if p.exists() else "ФАЙЛ НЕ НАЙДЕН")

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/repo/solution")

for f in p.iterdir():
    print(repr(f.name), "—", f.exists())

print("\nСодержимое правильного файла:")
f = p / "_init_.py"
if f.exists():
    print(f.read_text())
else:
    print("Файл _init_.py отсутствует")

In [ ]:
%cd /kaggle/working/repo

!python -c "import solution; import solution.rmsnorm as r; print('Пакет:', solution.__file__); print('Модуль:', r.file_); print('Функция в rmsnorm:', hasattr(r, 'patch_rmsnorm')); print('Функция в solution:', hasattr(solution, 'patch_rmsnorm')); print('Содержимое модуля:', [x for x in dir(r) if 'rms' in x.lower()])"

In [ ]:
%cd /kaggle/working/repo

!python -c "import solution; import importlib; r = importlib.import_module('solution.rmsnorm'); print('RMSNorm в модуле:', hasattr(r, 'patch_rmsnorm')); print('RMSNorm в пакете:', hasattr(solution, 'patch_rmsnorm')); print('Apply:', hasattr(solution, 'apply'))"

In [ ]:
import sys
import importlib
from pathlib import Path

p = Path("/kaggle/working/repo/solution/_init_.py")

p.write_text(
    "from .swiglu import patch_qwen2_mlp\n"
    "from .rmsnorm import patch_rmsnorm\n\n"
    "def apply():\n"
    "    patch_qwen2_mlp()\n"
    "    patch_rmsnorm()\n"
)

sys.modules.pop("solution", None)
sys.modules.pop("solution.rmsnorm", None)
sys.modules.pop("solution.swiglu", None)
importlib.invalidate_caches()

import solution

print("RMSNorm:", hasattr(solution, "patch_rmsnorm"))
print("Apply:", hasattr(solution, "apply"))

In [ ]:
from pathlib import Path

p = Path("/kaggle/working/repo/solution/_init_.py")

print("СОДЕРЖИМОЕ ФАЙЛА:")
print(repr(p.read_text()))

print("\nПРОВЕРКА RMSNORM:")
r = Path("/kaggle/working/repo/solution/rmsnorm.py")
print("Функция объявлена:", "def patch_rmsnorm" in r.read_text())

In [ ]:
import importlib
import solution

r = importlib.import_module("solution.rmsnorm")

print("Путь пакета:", solution._file_)
print("Путь RMSNorm:", r._file_)
print("Функция в модуле:", getattr(r, "patch_rmsnorm", "НЕТ"))
print("Функция в пакете:", getattr(solution, "patch_rmsnorm", "НЕТ"))
print("Импорт пакета:", solution._dict_.keys())

In [1]:
%cd /kaggle/working/repo

import solution
import importlib

r = importlib.import_module("solution.rmsnorm")

print("RMSNorm в модуле:", hasattr(r, "patch_rmsnorm"))
print("RMSNorm в пакете:", hasattr(solution, "patch_rmsnorm"))
print("Apply:", hasattr(solution, "apply"))

[Errno 2] No such file or directory: '/kaggle/working/repo'
/kaggle/working


ModuleNotFoundError: No module named 'solution'

In [2]:
import os

print("Текущая папка:", os.getcwd())
print("Содержимое /kaggle/working:")
print(os.listdir("/kaggle/working"))
print("Папка repo существует:", os.path.exists("/kaggle/working/repo"))

Текущая папка: /kaggle/working
Содержимое /kaggle/working:
['.virtual_documents']
Папка repo существует: False


In [3]:
%cd /kaggle/working

!git clone https://github.com/shinzo-imperor/soup-kernel-challenge.git repo

/kaggle/working
Cloning into 'repo'...


In [4]:
import os

print(os.listdir("/kaggle/working/repo"))
print("solution существует:",
      os.path.exists("/kaggle/working/repo/solution"))

['.git']
solution существует: False


In [5]:
%cd /kaggle/working/repo

!git remote -v
!git branch -a
!git status

/kaggle/working/repo
origin	https://github.com/shinzo-imperor/soup-kernel-challenge.git (fetch)
origin	https://github.com/shinzo-imperor/soup-kernel-challenge.git (push)
On branch main

No commits yet

nothing to commit (create/copy files and use "git add" to track)


In [6]:
%cd /kaggle/working

!git clone https://github.com/MakazhanAlpamys/soup-kernel-challenge.git starter

/kaggle/working
Cloning into 'starter'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 31 (delta 0), reused 31 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 48.04 KiB | 1.14 MiB/s, done.


In [7]:
import os

print(os.listdir("/kaggle/working/starter"))
print("solution:", os.path.exists("/kaggle/working/starter/solution"))

['tests', 'README.md', '.gitignore', 'task', 'LICENSE', 'solution', '.git', 'NOTICE', 'templates', 'harness']
solution: True


In [8]:
import os

p = "/kaggle/working/starter"

print("Файлы проекта:", os.listdir(p))
print("\nПапка solution:", os.path.exists(p + "/solution"))
print("Файл harness:", os.path.exists(p + "/harness/run_task.py"))

Файлы проекта: ['tests', 'README.md', '.gitignore', 'task', 'LICENSE', 'solution', '.git', 'NOTICE', 'templates', 'harness']

Папка solution: True
Файл harness: True


In [9]:
from pathlib import Path

p = Path("/kaggle/working/starter/solution")

for f in p.iterdir():
    print(f.name)

swiglu.py
__init__.py
test_kernels.py


In [10]:
from pathlib import Path

p = Path("/kaggle/working/starter/solution")

print("ФАЙЛЫ SOLUTION:")
for f in p.iterdir():
    if f.is_file():
        print(f"- {f.name}")

print("\nINIT.PY:")
print((p / "_init_.py").read_text())

ФАЙЛЫ SOLUTION:
- swiglu.py
- __init__.py
- test_kernels.py

INIT.PY:


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/starter/solution/_init_.py'

In [11]:
from pathlib import Path

p = Path("/kaggle/working/starter/solution")

print("ФАЙЛЫ:")
for f in p.iterdir():
    print(f.name)

print("\nСОДЕРЖИМОЕ _init_.py:")
print((p / "_init_.py").read_text())

ФАЙЛЫ:
swiglu.py
__init__.py
test_kernels.py

СОДЕРЖИМОЕ _init_.py:


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/starter/solution/_init_.py'

In [12]:
from pathlib import Path

p = Path("/kaggle/working/starter/solution")

print("Папка существует:", p.exists())
print("Файлы:", [repr(f.name) for f in p.iterdir()])

Папка существует: True
Файлы: ["'swiglu.py'", "'__init__.py'", "'test_kernels.py'"]


In [13]:
from pathlib import Path

p = Path("/kaggle/working/starter/solution/_init_.py")
print(p.read_text())

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/starter/solution/_init_.py'

In [14]:
from pathlib import Path

p = Path("/kaggle/working/starter/solution/_init_.py")

print("Файл найден:", p.exists())

if p.exists():
    print(p.read_text())

Файл найден: False


In [15]:
from pathlib import Path

for folder in [
    Path("/kaggle/working/starter/solution"),
    Path("/kaggle/working/repo/solution"),
]:
    print("\nПАПКА:", folder)
    print("Существует:", folder.exists())
    if folder.exists():
        print("Файлы:", [repr(x.name) for x in folder.iterdir()])


ПАПКА: /kaggle/working/starter/solution
Существует: True
Файлы: ["'swiglu.py'", "'__init__.py'", "'test_kernels.py'"]

ПАПКА: /kaggle/working/repo/solution
Существует: False


In [16]:
from pathlib import Path

p = Path("/kaggle/working/starter/solution/_init_.py")

p.write_text("""from .swiglu import patch_qwen2_mlp

def apply():
    patch_qwen2_mlp()
""")

print("Исходное подключение восстановлено.")
print(p.read_text())

Исходное подключение восстановлено.
from .swiglu import patch_qwen2_mlp

def apply():
    patch_qwen2_mlp()



In [17]:
from pathlib import Path

solution = Path("/kaggle/working/starter/solution")

rmsnorm_code = r'''
import torch
import triton
import triton.language as tl


@triton.jit
def rms_fwd(X, W, Y, INV,
            H: tl.constexpr, EPS: tl.constexpr,
            BLOCK: tl.constexpr):
    row = tl.program_id(0)
    col = tl.arange(0, BLOCK)

    x = tl.load(X + row * H + col, col < H, 0).to(tl.float32)
    w = tl.load(W + col, col < H, 0).to(tl.float32)

    mean_sq = tl.sum(x * x, 0) / H
    inv = tl.rsqrt(mean_sq + EPS)

    tl.store(Y + row * H + col, x * inv * w, col < H)
    tl.store(INV + row, inv)


@triton.jit
def rms_bwd(X, W, DY, INV, DX,
            H: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    col = tl.arange(0, BLOCK)

    x = tl.load(X + row * H + col, col < H, 0).to(tl.float32)
    w = tl.load(W + col, col < H, 0).to(tl.float32)
    dy = tl.load(DY + row * H + col, col < H, 0).to(tl.float32)
    inv = tl.load(INV + row)

    g = dy * w
    dot = tl.sum(g * x, 0) / H
    dx = inv * (g - x * inv * inv * dot)

    tl.store(DX + row * H + col, dx, col < H)


class RMSNormFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, weight, eps):
        shape = x.shape
        h = shape[-1]
        x2 = x.contiguous().view(-1, h)

        y = torch.empty_like(x2)
        inv = torch.empty(
            (x2.shape[0],), device=x.device, dtype=torch.float32
        )

        block = triton.next_power_of_2(h)
        rms_fwd[(x2.shape[0],)](
            x2, weight, y, inv, h, eps, block
        )

        ctx.save_for_backward(x2, weight, inv)
        ctx.shape = shape
        ctx.block = block
        return y.view(shape)

    @staticmethod
    def backward(ctx, grad_output):
        x, weight, inv = ctx.saved_tensors
        dy = grad_output.contiguous().view_as(x)
        dx = torch.empty_like(x)

        rms_bwd[(x.shape[0],)](
            x, weight, dy, inv, dx, x.shape[1], ctx.block
        )

        dw = (
            dy.float() * x.float() * inv[:, None]
        ).sum(dim=0).to(weight.dtype)

        return dx.view(ctx.shape), dw, None


def patch_rmsnorm():
    from transformers.models.qwen2 import modeling_qwen2

    cls = modeling_qwen2.Qwen2RMSNorm

    def forward(self, hidden_states):
        if not hidden_states.is_cuda:
            x = hidden_states.float()
            variance = x.pow(2).mean(-1, keepdim=True)
            return self.weight * (
                x * torch.rsqrt(variance + self.variance_epsilon)
            ).to(hidden_states.dtype)

        return RMSNormFunction.apply(
            hidden_states, self.weight, self.variance_epsilon
        )

    cls.forward = forward
'''

(solution / "rmsnorm.py").write_text(rmsnorm_code)

(solution / "_init_.py").write_text(
    "from .swiglu import patch_qwen2_mlp\n"
    "from .rmsnorm import patch_rmsnorm\n\n"
    "def apply():\n"
    "    patch_qwen2_mlp()\n"
    "    patch_rmsnorm()\n"
)

print("RMSNorm восстановлен.")
print("Файлы:", [f.name for f in solution.iterdir()])

RMSNorm восстановлен.
Файлы: ['swiglu.py', '__init__.py', '_init_.py', 'test_kernels.py', 'rmsnorm.py']


In [18]:
import sys
import importlib

sys.path.insert(0, "/kaggle/working/starter")

import solution
importlib.reload(solution)

r = importlib.import_module("solution.rmsnorm")
importlib.reload(r)

print("RMSNorm в модуле:", hasattr(r, "patch_rmsnorm"))
print("RMSNorm в пакете:", hasattr(solution, "patch_rmsnorm"))
print("Apply:", hasattr(solution, "apply"))
print("Triton forward:", hasattr(r, "rms_fwd"))
print("Triton backward:", hasattr(r, "rms_bwd"))

RMSNorm в модуле: True
RMSNorm в пакете: False
Apply: True
Triton forward: True
Triton backward: True


In [19]:
import importlib
import torch
import solution

r = importlib.import_module("solution.rmsnorm")
importlib.reload(r)
importlib.reload(solution)

print("Патч доступен:", hasattr(solution, "patch_rmsnorm"))

# Проверка точности на GPU
for dtype in (torch.float32, torch.float16):
    x = torch.randn((8, 1536), device="cuda", dtype=dtype)
    w = torch.randn((1536,), device="cuda", dtype=dtype)

    xc = x.detach().clone().requires_grad_(True)
    wc = w.detach().clone().requires_grad_(True)
    xr = x.detach().clone().requires_grad_(True)
    wr = w.detach().clone().requires_grad_(True)

    y_custom = r.RMSNormFunction.apply(xc, wc, 1e-6)

    xf = xr.float()
    y_ref = wr * (
        xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + 1e-6)
    ).to(dtype)

    dy = torch.randn_like(y_ref)
    y_custom.backward(dy)
    y_ref.backward(dy)

    print(
        dtype,
        "forward:", (y_custom - y_ref).abs().max().item(),
        "dx:", (xc.grad - xr.grad).abs().max().item(),
        "dw:", (wc.grad - wr.grad).abs().max().item()
    )

Патч доступен: False
torch.float32 forward: 9.5367431640625e-07 dx: 9.5367431640625e-07 dw: 1.1920928955078125e-06
torch.float16 forward: 0.00390625 dx: 0.00390625 dw: 0.0078125


In [20]:
from pathlib import Path
import importlib
import sys

sys.path.insert(0, "/kaggle/working/starter")

init_file = Path("/kaggle/working/starter/solution/_init_.py")
init_file.write_text(
    "from .swiglu import patch_qwen2_mlp\n"
    "from .rmsnorm import patch_rmsnorm\n"
    "\n"
    "def apply():\n"
    "    patch_qwen2_mlp()\n"
    "    patch_rmsnorm()\n"
)

import solution
importlib.reload(solution)

print("Патч доступен:", hasattr(solution, "patch_rmsnorm"))
print("Apply доступен:", hasattr(solution, "apply"))

Патч доступен: False
Apply доступен: True


In [22]:
import solution
import sys
from pathlib import Path

print("Пакет загружен из:", solution.__file__)
print("Путь starter существует:", Path(
    "/kaggle/working/starter/solution/_init_.py"
).exists())

print("\nСодержимое _init_.py:")
print(Path("/kaggle/working/starter/solution/_init_.py").read_text())

print("\nПуть в sys.path:")
print(sys.path[:5])

Пакет загружен из: /kaggle/working/starter/solution/__init__.py
Путь starter существует: True

Содержимое _init_.py:
from .swiglu import patch_qwen2_mlp
from .rmsnorm import patch_rmsnorm

def apply():
    patch_qwen2_mlp()
    patch_rmsnorm()


Путь в sys.path:
['/kaggle/working/starter', '/kaggle/working/starter', '/kaggle/working', '/kaggle/lib/kagglegym', '/kaggle/lib']


In [25]:
import solution
import solution.rmsnorm as r

print("solution:", solution.__file__)
print("rmsnorm:", r.__file__)
print("Функция в rmsnorm:", hasattr(r, "patch_rmsnorm"))
print("Функция в solution:", hasattr(solution, "patch_rmsnorm"))
print("Атрибуты solution:", [x for x in dir(solution) if "rms" in x.lower()])
print("Атрибуты rmsnorm:", [x for x in dir(r) if "rms" in x.lower()])

solution: /kaggle/working/starter/solution/__init__.py
rmsnorm: /kaggle/working/starter/solution/rmsnorm.py
Функция в rmsnorm: True
Функция в solution: False
Атрибуты solution: ['rmsnorm']
Атрибуты rmsnorm: ['RMSNormFunction', 'patch_rmsnorm', 'rms_bwd', 'rms_fwd']


In [26]:
from pathlib import Path
import importlib
import solution

p = Path("/kaggle/working/starter/solution/_init_.py")
p.write_text(
    "from . import swiglu, rmsnorm\n"
    "\n"
    "def apply():\n"
    "    swiglu.patch_qwen2_mlp()\n"
    "    rmsnorm.patch_rmsnorm()\n"
)

importlib.reload(solution)

print("RMSNorm доступен:", hasattr(solution.rmsnorm, "patch_rmsnorm"))
print("Apply вызывает оба патча:")
import inspect
print(inspect.getsource(solution.apply))

RMSNorm доступен: True
Apply вызывает оба патча:
def apply():
    patch_qwen2_mlp()  # the example: works out of the box, does not count as your own kernel



In [27]:
from pathlib import Path

p = Path("/kaggle/working/starter/solution/_init_.py")

p.write_text("""from . import swiglu, rmsnorm

def apply():
    swiglu.patch_qwen2_mlp()
    rmsnorm.patch_rmsnorm()
""")

print("Файл обновлён!")

Файл обновлён!


In [28]:
import importlib
import solution

importlib.invalidate_caches()
importlib.reload(solution)

import inspect
print(inspect.getsource(solution.apply))

def apply():
    patch_qwen2_mlp()  # the example: works out of the box, does not count as your own kernel



In [29]:
from pathlib import Path
import importlib
import sys
import shutil

folder = Path("/kaggle/working/starter/solution")
init_file = folder / "_init_.py"

init_file.write_text(
    "from . import swiglu, rmsnorm\n\n"
    "def apply():\n"
    "    swiglu.patch_qwen2_mlp()\n"
    "    rmsnorm.patch_rmsnorm()\n"
)

shutil.rmtree(folder / "_pycache_", ignore_errors=True)

importlib.invalidate_caches()
importlib.reload(sys.modules["solution"])

print(init_file.read_text())
print("Функции apply:", sys.modules["solution"].apply._code_.co_names)

from . import swiglu, rmsnorm

def apply():
    swiglu.patch_qwen2_mlp()
    rmsnorm.patch_rmsnorm()



AttributeError: 'function' object has no attribute '_code_'

In [30]:
import os, sys, subprocess, pathlib

ROOT = "/kaggle/working/starter"
os.chdir(ROOT)

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=ROOT)
    return (r.stdout + r.stderr).strip()

print("=== 1. Папка и git ===")
print(sh("ls -la . solution; echo; git status --short; echo; git log --oneline -3"))

print("\n=== 2. Что физически лежит в solution/__init__.py ===")
p = pathlib.Path(ROOT, "solution", "__init__.py")
print("размер:", p.stat().st_size, "байт")
print(p.read_text())

print("\n=== 3. Где вне solution/ упоминается solution / apply ===")
print(sh(r"grep -rnE 'solution|apply\(' . --include='*.py' --include='*.sh' --include='*.md' --include='*.yaml' --include='*.yml' --include='*.toml' --exclude-dir=.git --exclude-dir=solution | head -80"))

print("\n=== 4. Свежий процесс Python (так, как его запустит harness) ===")
code = (
    "import sys, inspect, solution\n"
    "print('файл пакета:', solution.__file__)\n"
    "print('есть apply:', hasattr(solution, 'apply'))\n"
    "print(inspect.getsource(solution.apply))\n"
)
r = subprocess.run([sys.executable, "-B", "-c", code], cwd=ROOT, capture_output=True, text=True)
print(r.stdout + r.stderr)

print("=== 5. Другие копии solution на диске / установленные пакеты ===")
print(sh("find /kaggle/working /usr/local/lib /opt -type d -name solution 2>/dev/null | head -20"))
print(sh("pip list 2>/dev/null | grep -iE 'solution|soup|decentr' || echo 'пакетов с таким именем нет'"))
print(sh("ls /kaggle/working"))

print("\n=== 6. Что загружено в самом ядре ноутбука ===")
m = sys.modules.get("solution")
print("solution в sys.modules:", m)
if m is not None:
    ap = getattr(m, "apply", None)
    print("apply:", ap)
    if ap is not None:
        print("  определена в:", ap.__code__.co_filename, "строка", ap.__code__.co_firstlineno)
print("sys.path[:6]:", sys.path[:6])

=== 1. Папка и git ===
.:
total 68
drwxr-xr-x 8 root root  4096 Oct 31 00:32 .
drwxr-xr-x 5 root root  4096 Oct 31 00:32 ..
drwxr-xr-x 8 root root  4096 Oct 31 00:32 .git
-rw-r--r-- 1 root root    74 Oct 31 00:32 .gitignore
drwxr-xr-x 2 root root  4096 Oct 31 00:32 harness
-rw-r--r-- 1 root root 11358 Oct 31 00:32 LICENSE
-rw-r--r-- 1 root root   835 Oct 31 00:32 NOTICE
-rw-r--r-- 1 root root 13825 Oct 31 00:32 README.md
drwxr-xr-x 3 root root  4096 Oct 31 00:40 solution
drwxr-xr-x 2 root root  4096 Oct 31 00:32 task
drwxr-xr-x 2 root root  4096 Oct 31 00:32 templates
drwxr-xr-x 2 root root  4096 Oct 31 00:32 tests

solution:
total 32
drwxr-xr-x 3 root root 4096 Oct 31 00:40 .
drwxr-xr-x 8 root root 4096 Oct 31 00:32 ..
-rw-r--r-- 1 root root 1030 Oct 31 00:32 __init__.py
-rw-r--r-- 1 root root  101 Oct 31 00:51 _init_.py
drwxr-xr-x 2 root root 4096 Oct 31 00:40 __pycache__
-rw-r--r-- 1 root root 2586 Oct 31 00:40 rmsnorm.py
-rw-r--r-- 1 root root 3185 Oct 31 00:32 swiglu.py
-rw-r--r--

In [31]:
import pathlib, subprocess, sys

SOL = pathlib.Path("/kaggle/working/starter/solution")

# 1. Правим настоящий __init__.py (по два подчёркивания с каждой стороны)
init = SOL / "__init__.py"
src = init.read_text()
old_import = "from .swiglu import patch_qwen2_mlp"
old_call = "    patch_qwen2_mlp()  # the example: works out of the box, does not count as your own kernel"
assert old_import in src and old_call in src, "в __init__.py не тот текст, который ожидался"
src = src.replace(old_import, "from . import swiglu, rmsnorm")
src = src.replace(
    old_call,
    "    swiglu.patch_qwen2_mlp()  # the example, does not count as own kernel\n"
    "    rmsnorm.patch_rmsnorm()  # own Triton RMSNorm",
)
init.write_text(src)

# 2. Удаляем лишний файл с неправильным именем
stray = SOL / "_init_.py"
if stray.exists():
    stray.unlink()

print(subprocess.run(f"ls -la {SOL}", shell=True, capture_output=True, text=True).stdout)

# 3. Проверка в чистом процессе
check = '''
import inspect, solution
print("apply в пакете:")
print(inspect.getsource(solution.apply))

from transformers.models.qwen2 import modeling_qwen2 as m
before = m.Qwen2RMSNorm.forward
solution.apply()
after = m.Qwen2RMSNorm.forward
print("forward RMSNorm заменён:", before is not after)
print("определён в:", after.__code__.co_filename)

import torch
norm = m.Qwen2RMSNorm(1536, eps=1e-6).cuda().half()
x = torch.randn(4, 512, 1536, device="cuda", dtype=torch.float16, requires_grad=True)
y = norm(x)
y.float().sum().backward()
print("forward и backward выполнились:", y.dtype, tuple(y.shape), "grad:", x.grad.dtype)
'''
r = subprocess.run([sys.executable, "-B", "-c", check],
                   cwd="/kaggle/working/starter", capture_output=True, text=True)
print(r.stdout + r.stderr[-3000:])

total 28
drwxr-xr-x 3 root root 4096 Oct 31 01:01 .
drwxr-xr-x 8 root root 4096 Oct 31 00:32 ..
-rw-r--r-- 1 root root 1054 Oct 31 01:01 __init__.py
drwxr-xr-x 2 root root 4096 Oct 31 00:40 __pycache__
-rw-r--r-- 1 root root 2586 Oct 31 00:40 rmsnorm.py
-rw-r--r-- 1 root root 3185 Oct 31 00:32 swiglu.py
-rw-r--r-- 1 root root 1300 Oct 31 00:32 test_kernels.py

apply в пакете:
def apply():
    swiglu.patch_qwen2_mlp()  # the example, does not count as own kernel
    rmsnorm.patch_rmsnorm()  # own Triton RMSNorm

forward RMSNorm заменён: True
определён в: /kaggle/working/starter/solution/rmsnorm.py
forward и backward выполнились: torch.float16 (4, 512, 1536) grad: torch.float16



In [32]:
import subprocess

def show(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="/kaggle/working/starter")
    print(r.stdout + r.stderr)

print("=== README, строки 95-115 ===")
show("sed -n '95,115p' README.md")
print("=== harness/_train.py, строки 40-135 ===")
show("sed -n '40,135p' harness/_train.py")

=== README, строки 95-115 ===
  logits, and Soup writes each row's loss to `rewind.jsonl`. These are Soup outputs: keep them
  with the same values (you may compute them faster).

Places to hook in `apply()` (all looked up at call time):

- model classes, e.g. `transformers.models.qwen2.modeling_qwen2.Qwen2MLP.forward` (the example);
- `soup_cli.trainer.sft.SFTTrainerWrapper._setup_transformers`: when it returns, `self.model` is
  the final 4-bit PEFT model, before the trainer is built;
- `soup_cli.trainer.sft.SFTTrainerWrapper.train`: the trainer and the model exist, training has
  not started;
- `trl.SFTTrainer` (Soup builds its trainer class from it during setup), peft and bitsandbytes
  functions.

Patch narrowly. `apply()` also runs before Soup's regression tests (check 10), where some tests
call these methods with fake transformers/peft modules; a patch that assumes a real Qwen model
everywhere fails them. Guard it (`if not isinstance(self.model, torch.nn.Module): return ...`) an

In [33]:
!cd /kaggle/working/starter && python harness/run_task.py --only solution --out runs_try 2>&1 | tail -80

/kaggle/working/starter/data/train.jsonl is missing: run `python task/prepare_data.py` first


In [34]:
import json, pathlib, subprocess

ROOT = pathlib.Path("/kaggle/working/starter")

def show(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=ROOT)
    print(r.stdout + r.stderr)

print("=== README, строки 125-150 ===")
show("sed -n '125,150p' README.md")
print("=== README, строки 175-192 ===")
show("sed -n '175,192p' README.md")

p = ROOT / "runs_try" / "solution_profile" / "profile.json"
print("=== Профиль:", p, "===")
if not p.exists():
    show("find runs_try -maxdepth 3 | head -40")
else:
    data = json.loads(p.read_text())
    rows = None
    if isinstance(data, list):
        rows = data
    elif isinstance(data, dict):
        print("ключи верхнего уровня:", list(data.keys()))
        for v in data.values():
            if isinstance(v, list) and v and isinstance(v[0], dict) and "name" in v[0]:
                rows = v
                break
    if rows:
        total = sum(r.get("time_us", 0) for r in rows)
        print(f"всего GPU-времени по ядрам: {total/1000:.1f} мс")
        for r in rows[:25]:
            print(f"{r['time_us']/total*100:5.1f}%  {r['time_us']/1000:9.1f} мс  x{r['count']:<6} {r['name'][:90]}")
    else:
        print(str(data)[:3000])

=== README, строки 125-150 ===
| `harness/common.py`, `harness/example_reference.py` | shared settings, a frozen copy of the example | no |
| `tests/` | Soup's regression tests: 79 tests in five files | no |
| `templates/` | Kaggle notebook, `SUBMISSION.md` | copy and fill |

## What is measured

- **Time**: the wall time of the whole `soup train` command, from start to saved adapter, timed
  from outside the process. Triton compilation and autotuning count: every run starts with empty
  compiler caches. Base and solution run in the same session; the score uses the ratio.
- **Memory**: peak GPU memory, sampled by `nvidia-smi` from outside the process every 100 ms and
  summed over all GPUs. Both sides run with `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`,
  so the peak follows the tensors that are alive rather than allocator fragmentation.
- **Quality**: `harness/eval_adapter.py`, the mean loss on the answer tokens of held-out rows. It
  loads your adapter with plain transformers

In [35]:
!cd /kaggle/working/starter && SOUP_APPLY_SOLUTION=1 python -m pytest tests -q 2>&1 | tail -30

    from soup_cli.data.formats import (
E   ModuleNotFoundError: No module named 'soup_cli'
____________________ ERROR collecting tests/test_loader.py _____________________
ImportError while importing test module '/kaggle/working/starter/tests/test_loader.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_loader.py:10: in <module>
    from soup_cli.data.loader import load_raw_data
E   ModuleNotFoundError: No module named 'soup_cli'
_________________ ERROR collecting tests/test_trainer_init.py __________________
ImportError while importing test module '/kaggle/working/starter/tests/test_trainer_init.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _

In [36]:
import subprocess

def show(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="/kaggle/working/starter")
    print(r.stdout + r.stderr)

print("=== Python и установленные пакеты ===")
show("python --version; pip list 2>/dev/null | grep -iE 'soup|trl|peft|bitsandbytes|transformers|triton|^torch ' ; which soup || echo 'команды soup нет'")

print("=== Где в проекте описана установка ===")
show("grep -rnEi 'pip install|soup-cli|soup_cli|requirements' README.md templates task harness/common.py harness/run_task.py 2>/dev/null | cut -c1-220 | head -40")

print("=== README, строки 25-65 ===")
show("sed -n '25,65p' README.md")

print("=== Файлы в templates/ и task/ ===")
show("ls -la templates task")

print("=== Есть ли интернет (код 200/301 = есть, 000 = нет) ===")
show("curl -s -o /dev/null -m 8 -w '%{http_code}\\n' https://pypi.org || echo 'нет доступа'")

=== Python и установленные пакеты ===
Python 3.13.15
beautifulsoup4                        4.13.5
peft                                  0.20.0
sentence-transformers                 5.7.0
soupsieve                             2.9.2
torch                                 2.11.0+cu128
transformers                          5.16.1
triton                                3.6.0
команды soup нет

=== Где в проекте описана установка ===
README.md:10:`soup train` with `soup-cli==0.77.0` fine-tunes **Qwen2.5-1.5B-Instruct** (4-bit NF4 weights,
README.md:58:pip install "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" \
README.md:101:- `soup_cli.trainer.sft.SFTTrainerWrapper._setup_transformers`: when it returns, `self.model` is
README.md:103:- `soup_cli.trainer.sft.SFTTrainerWrapper.train`: the trainer and the model exist, training has
README.md:151:triton, transformers, peft, trl, bitsandbytes and `soup-cli==0.77.0`; ideas from other kernels
templates/kaggle_notebook.ipy

In [37]:
!python -m pip install -q "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest
!python -c "import torch, triton, transformers, peft, trl, bitsandbytes, soup_cli.cli; print('torch', torch.__version__, '| triton', triton.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| trl', trl.__version__, '| bnb', bitsandbytes.__version__)"
!cd /kaggle/working/starter && python task/prepare_data.py 2>&1 | tail -15 && ls data

torch 2.11.0+cu128 | triton 3.6.0 | transformers 5.16.1 | peft 0.20.0 | trl 0.29.1 | bnb 0.50.2
Generating train split: 52201 examples [00:01, 27895.20 examples/s]
wrote 256 training rows and 256 eval rows to /kaggle/working/starter/data
eval_public.jsonl  train.jsonl


In [38]:
!cd /kaggle/working/starter && python harness/run_task.py --only solution --profile --out runs_try 2>&1 | tail -80

[run_task] warm-up done in 21.07 s
[run_task] solution_1: training ...
[run_task] solution_1: ok, 88.72 s, peak 12506 MiB
[run_task] eval solution ...
[run_task] solution_profile: training ...
[run_task] solution_profile: ok, 122.04 s, peak 12518 MiB

=== result ===
solution  time     88.7 s   peak  12506 MiB   eval loss 2.0497
profiler table: /kaggle/working/starter/runs_try/solution_profile/profile.json
written: /kaggle/working/starter/runs_try/result.json


In [39]:
import json, pathlib, subprocess

ROOT = pathlib.Path("/kaggle/working/starter")

r = ROOT / "runs_try" / "result.json"
print("=== result.json ===")
print(r.read_text()[:3000] if r.exists() else "нет файла")

p = ROOT / "runs_try" / "solution_profile" / "profile.json"
print("\n=== Профиль ===")
if not p.exists():
    print(subprocess.run("find runs_try -maxdepth 3 | head -40", shell=True,
                         capture_output=True, text=True, cwd=ROOT).stdout)
else:
    data = json.loads(p.read_text())
    rows = None
    if isinstance(data, list):
        rows = data
    elif isinstance(data, dict):
        print("ключи верхнего уровня:", list(data.keys()))
        for v in data.values():
            if isinstance(v, list) and v and isinstance(v[0], dict) and "name" in v[0]:
                rows = v
                break
    if rows:
        total = sum(x.get("time_us", 0) for x in rows)
        print(f"всего GPU-времени по ядрам: {total/1000:.1f} мс")
        for x in rows[:25]:
            print(f"{x['time_us']/total*100:5.1f}%  {x['time_us']/1000:9.1f} мс  x{x['count']:<6} {x['name'][:90]}")
    else:
        print(str(data)[:3000])

=== result.json ===
{
  "task": {
    "train_file": "/kaggle/working/starter/data/train.jsonl",
    "eval_file": "/kaggle/working/starter/data/eval_public.jsonl",
    "seed": 1,
    "repeats": 1,
    "only": "solution"
  },
  "versions": {
    "python": "3.13.15",
    "soup-cli": "0.77.0",
    "torch": "2.11.0+cu128",
    "triton": "3.6.0",
    "transformers": "5.16.1",
    "peft": "0.20.0",
    "trl": "0.29.1",
    "bitsandbytes": "0.50.2",
    "accelerate": "1.14.0",
    "datasets": "4.8.5",
    "gpus": [
      "Tesla T4, 580.178.04",
      "Tesla T4, 580.178.04"
    ]
  },
  "packages_changed": [],
  "summary": {
    "solution": {
      "ok": true,
      "runs": 1,
      "wall_s": 88.72,
      "gpu_peak_mib": 12506,
      "train_loop_s": 56.06,
      "eval_loss": 2.0497
    }
  },
  "runs": {
    "solution": [
      {
        "run": "solution_1",
        "side": "solution",
        "exit_code": 0,
        "wall_s": 88.72,
        "gpu_peak_mib": 12506,
        "gpu_start_mib": 126,


In [40]:
!cd /kaggle/working/starter && SOUP_APPLY_SOLUTION=1 python -m pytest tests -q 2>&1 | tail -30

........................................................................ [ 91%]
.......                                                                  [100%]
=============================== warnings summary ===============================
../../../usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:365: 14 warnings
  /usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:365: DeprecationWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
79 passed, 14 warnings in 1.83s


In [41]:
import inspect, subprocess, os

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="/kaggle/working/starter")
    return (r.stdout + r.stderr).strip()

def show(title, text, limit=4500):
    print(f"\n=== {title} ===")
    print(text[:limit])

import bitsandbytes as bnb
import bitsandbytes.functional as F
show("версия bitsandbytes", bnb.__version__)
show("F.dequantize_4bit", inspect.getsource(F.dequantize_4bit))
try:
    from bitsandbytes.autograd import _functions as AF
    show("MatMul4Bit", inspect.getsource(AF.MatMul4Bit))
except Exception as e:
    show("MatMul4Bit", "не удалось: " + repr(e))
show("Linear4bit.forward", inspect.getsource(bnb.nn.Linear4bit.forward), 3000)
bnb_dir = os.path.dirname(bnb.__file__)
show("где в bitsandbytes встречается dequantize_4bit",
     sh(f"grep -rn 'dequantize_4bit' {bnb_dir} --include='*.py' | cut -c1-170 | head -30"), 4000)
show("task/task.yaml", sh("cat task/task.yaml"))
import soup_cli
soup_dir = os.path.dirname(soup_cli.__file__)
show("Soup: где считаются loss / entropy / accuracy",
     sh(f"grep -rnE 'entropy|accuracy|rewind|compute_loss|log_softmax|cross_entropy' {soup_dir}/trainer | cut -c1-190 | head -60"), 7000)


=== версия bitsandbytes ===
0.50.2

=== F.dequantize_4bit ===
def dequantize_4bit(
    A: torch.Tensor,
    quant_state: Optional[QuantState] = None,
    absmax: Optional[torch.Tensor] = None,
    out: Optional[torch.Tensor] = None,
    blocksize: Optional[int] = None,
    quant_type="fp4",
) -> torch.Tensor:
    """Dequantizes a packed 4-bit quantized tensor.

    The input tensor is dequantized by dividing it into blocks of `blocksize` values.
    The absolute maximum value within these blocks is used for scaling
    the non-linear dequantization.

    Args:
        A (`torch.Tensor`): The quantized input tensor.
        quant_state ([`QuantState`], *optional*):
            The quantization state as returned by [`quantize_4bit`].
            Required if `absmax` is not provided.
        absmax (`torch.Tensor`, *optional*):
            A tensor containing the scaling values.
            Required if `quant_state` is not provided and ignored otherwise.
        out (`torch.Tensor`, *opt

In [42]:
!cd /kaggle/working/starter && SOUP_APPLY_SOLUTION=1 python -m pytest tests -q 2>&1 | tail -30

........................................................................ [ 91%]
.......                                                                  [100%]
=============================== warnings summary ===============================
../../../usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:365: 14 warnings
  /usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:365: DeprecationWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
79 passed, 14 warnings in 1.74s


In [44]:
!npm install -g @anthropic-ai/claude-code

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 2 packages in 4s
⠸npm notice
npm notice New major version of npm available! 10.8.2 -> 12.2.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.2.1
npm notice To update run: npm install -g npm@12.2.1
npm notice
⠸

In [45]:
!claude --version

2.1.197 (Claude Code)


In [50]:
%cd /kaggle/working/starter
!claude

/kaggle/working
78WelcometoClaudeCodev2.1.197

[>0qLet'sgetstarted.

Choosethetextstylethatlooksbestwithyourterminal
Tochangethislater,run/theme

1.Auto(matchterminal)
❯2.Darkmode✔
3.Lightmode
4.Darkmode(colorblind-friendly)
5.Lightmode(colorblind-friendly)
6.Darkmode(ANSIcolorsonly)
7.Lightmode(ANSIcolorsonly)

╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌
 1 function greet(){
 2 -  console.log("Hello, World!");                                           
 2 +  console.log("Hello, Claude!");                                          
 3 }
╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌
 Syntaxtheme:MonokaiExtended(ctrl+ttodisable)
(B
PressCtrl-Cagaintoexit



In [ ]:
import os, json, subprocess
from kaggle_secrets import UserSecretsClient
os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
json.dump({"hasCompletedOnboarding": True, "theme": "dark"},
          open(os.path.expanduser("~/.claude.json"), "w"))
r = subprocess.run(["claude", "-p", "скажи привет"], capture_output=True,
                   text=True, timeout=120, cwd="/kaggle/working/starter")
print("EXIT:", r.returncode)
print("OUT:", r.stdout)
print("ERR:", r.stderr)

In [53]:
!cat /kaggle/working/starter/solution/__init__.py
!cat /kaggle/working/starter/solution/rmsnorm.py

"""Your solution.

The harness imports this package and calls `apply()` once, in the same process as `soup train`,
before Soup builds the model and the trainer. Patch whatever you need (transformers, peft,
bitsandbytes, Soup itself) so the training run uses your Triton kernels.

The rules in short (the README and the task statement have the full ones):
- your own `@triton.jit` kernels, defined in this folder, run in every measured run and together
  take >= 5% of the training time; the example's kernels (however renamed or rewritten) and
  kernels generated by torch.compile do not count;
- no ready-made kernel libraries (unsloth, liger-kernel, xformers, flash-attn, torchao, ...);
- the task stays the same: model, data, batches and their order, steps, LoRA, optimizer, Soup's
  settings and outputs; no other GPU, no background processes, no tricks with the measurement.
"""

from . import swiglu, rmsnorm


def apply():
    swiglu.patch_qwen2_mlp()  # the example, does not count as own ker

In [55]:
!find /kaggle/working/starter -name "_pycache_" -type d -exec rm -rf {} +
%cd /kaggle/working/starter
!python -c "import solution; print(solution.__file__); solution.apply(); from transformers.models.qwen2 import modeling_qwen2 as m; print(m.Qwen2RMSNorm.forward.module, m.Qwen2RMSNorm.forward.qualname_)"

/kaggle/working
/kaggle/working/starter/solution/__init__.py
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import solution; print(solution.__file__); solution.apply(); from transformers.models.qwen2 import modeling_qwen2 as m; print(m.Qwen2RMSNorm.forward.module, m.Qwen2RMSNorm.forward.qualname_)
                                                                                                                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'function' object has no attribute 'module'


In [57]:
!find /kaggle/working/starter -name "_pycache_" -type d -exec rm -rf {} +
%cd /kaggle/working/starter
!python -c "import solution; print(solution.__file__); solution.apply(); from transformers.models.qwen2 import modeling_qwen2 as m; print(m.Qwen2RMSNorm.forward.qualname_)"

/kaggle/working
/kaggle/working/starter/solution/__init__.py
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import solution; print(solution.__file__); solution.apply(); from transformers.models.qwen2 import modeling_qwen2 as m; print(m.Qwen2RMSNorm.forward.qualname_)
                                                                                                                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'function' object has no attribute 'qualname_'


In [60]:
%cd /kaggle/working/starter
!python -c "import solution; solution.apply(); from transformers.models.qwen2 import modeling_qwen2 as m; print(m.Qwen2RMSNorm.forward.__module__, m.Qwen2RMSNorm.forward.__qualname__)"

/kaggle/working
solution.rmsnorm patch_rmsnorm.<locals>.forward


In [61]:
%cd /kaggle/working/starter
!python -c "import solution, inspect; solution.apply(); from transformers.models.qwen2 import modeling_qwen2 as m; print(inspect.getsourcefile(m.Qwen2RMSNorm.forward))"

/kaggle/working
/kaggle/working/starter/solution/rmsnorm.py


In [62]:
!grep -n -i -E "soup|pytest|triton|watch" /kaggle/working/starter/README.md | head -40

1:# Soup Kernel Challenge
3:Make `soup train` faster and lighter on a free Kaggle T4 with your own Triton kernels.
4:Case of [Soup](https://github.com/MakazhanAlpamys/Soup) at Decentrathon 6.0,
10:`soup train` with `soup-cli==0.77.0` fine-tunes **Qwen2.5-1.5B-Instruct** (4-bit NF4 weights,
14:changing what it trains. Your own `@triton.jit` kernels must do real work in it.
16:Base Soup on a Kaggle T4 (our measurement; yours comes from `harness/run_task.py`):
18:| | Base Soup 0.77.0 |
20:| Whole `soup train` command | about 79–83 s (56–61 s of it are the 64 training steps) |
24:Where the GPU time of a training step goes (our torch.profiler run of base Soup): matrix multiplies about
39:   git fetch https://github.com/MakazhanAlpamys/soup-kernel-challenge.git main
41:   git commit -m "Soup Kernel Challenge starter kit"
58:pip install "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" \
59:    "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest
61:python -m pytest 

In [63]:
%cd /kaggle/working/starter
!python -m pytest solution -q 2>&1 | tail -15

/kaggle/working
....                                                                     [100%]
4 passed in 4.92s


In [64]:
%cd /kaggle/working/starter
!python harness/run_task.py --help 2>&1 | head -30
!sed -n 55,70p README.md
!sed -n 128,148p README.md

/kaggle/working
usage: run_task.py [-h] [--only {base,solution}] [--repeats REPEATS]
                   [--profile] [--train-file TRAIN_FILE]
                   [--eval-file EVAL_FILE] [--seed SEED] [--out OUT]
                   [--timeout TIMEOUT]

Run the task and measure it: base Soup and your solution, one after the other
in this session.

options:
  -h, --help            show this help message and exit
  --only {base,solution}
  --repeats REPEATS
  --profile             also run your solution once under torch.profiler (for
                        your analysis)
  --train-file TRAIN_FILE
  --eval-file EVAL_FILE
  --seed SEED
  --out OUT
  --timeout TIMEOUT     seconds per run
Locally on a Linux machine with a CUDA GPU:

```bash
pip install "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" \
    "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest
python task/prepare_data.py          # data/train.jsonl, data/eval_public.jsonl
python -m pytest solution -q  

In [65]:
%cd /kaggle/working/starter
!ls data 2>&1 | head

/kaggle/working
eval_public.jsonl
train.jsonl


In [66]:
%cd /kaggle/working/starter
!ls runs data
!python -c "import json; d=json.load(open('runs/result.json')); print(json.dumps(d, indent=1)[:3000])"

/kaggle/working
ls: cannot access 'runs': No such file or directory
data:
eval_public.jsonl  train.jsonl
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import json; d=json.load(open('runs/result.json')); print(json.dumps(d, indent=1)[:3000])
                             ~~~~^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'runs/result.json'


In [67]:
!python harness/run_task.py --only solution --out runs/sol1 2>&1 | tail -40

[run_task] warm-up done in 13.53 s
[run_task] solution_1: training ...
[run_task] solution_1: ok, 77.18 s, peak 12506 MiB
[run_task] eval solution ...

=== result ===
solution  time     77.2 s   peak  12506 MiB   eval loss 2.04964
written: /kaggle/working/starter/runs/sol1/result.json


In [69]:
%cd /kaggle/working/starter
!python -c "import json; d=json.load(open('runs/sol1/result.json')); print(json.dumps(d, indent=1)[:3500])"

/kaggle/working
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import json; d=json.load(open('runs/sol1/result.json')); print(json.dumps(d, indent=1)[:3500])
                             ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'runs/sol1/result.json'


In [70]:
!ls -la /kaggle/working/starter/runs/sol1
!python -c "import json; d=json.load(open('/kaggle/working/starter/runs/sol1/result.json')); print(json.dumps(d, indent=1)[:3500])"

total 24
drwxr-xr-x 3 root root 4096 Oct 31 02:22 .
drwxr-xr-x 3 root root 4096 Oct 31 02:18 ..
-rw-r--r-- 1 root root  215 Oct 31 02:22 eval_solution.json
-rw-r--r-- 1 root root 1505 Oct 31 02:22 eval_solution.log
drwxr-xr-x 5 root root 4096 Oct 31 02:22 solution_1
-rw-r--r-- 1 root root  264 Oct 31 02:20 warmup.log
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import json; d=json.load(open('/kaggle/working/starter/runs/sol1/result.json')); print(json.dumps(d, indent=1)[:3500])
                             ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/starter/runs/sol1/result.json'


In [71]:
!find /kaggle/working/starter/runs -type f | head -50
!grep -r -i -l -E "share|triton" /kaggle/working/starter/runs | head

/kaggle/working/starter/runs/sol1/solution_1/adapter/README.md
/kaggle/working/starter/runs/sol1/solution_1/adapter/tokenizer_config.json
/kaggle/working/starter/runs/sol1/solution_1/adapter/adapter_config.json
/kaggle/working/starter/runs/sol1/solution_1/adapter/adapter_model.safetensors
/kaggle/working/starter/runs/sol1/solution_1/adapter/chat_template.jinja
/kaggle/working/starter/runs/sol1/solution_1/adapter/tokenizer.json
/kaggle/working/starter/runs/sol1/solution_1/adapter/checkpoint-64/README.md
/kaggle/working/starter/runs/sol1/solution_1/adapter/checkpoint-64/optimizer.pt
/kaggle/working/starter/runs/sol1/solution_1/adapter/checkpoint-64/tokenizer_config.json
/kaggle/working/starter/runs/sol1/solution_1/adapter/checkpoint-64/adapter_config.json
/kaggle/working/starter/runs/sol1/solution_1/adapter/checkpoint-64/adapter_model.safetensors
/kaggle/working/starter/runs/sol1/solution_1/adapter/checkpoint-64/chat_template.jinja
/kaggle/working/starter/runs/sol1/solution_1/adapter/che

In [72]:
!cat /kaggle/working/starter/runs/sol1/solution_1/metrics.json | head -c 3000
!ls /kaggle/working/starter/runs/sol1/solution_1/triton_cache/*/ | grep -E "rms_(fwd|bwd)\.cubin"

{
  "exit_code": 0,
  "solution": true,
  "apply_s": 2.84,
  "steps": 64,
  "train_loop_s": 57.04,
  "median_step_s": 0.8615,
  "step_s": [
    3.9933,
    0.751,
    1.1,
    1.1103,
    1.0791,
    0.7026,
    1.0792,
    1.1189,
    0.8384,
    0.701,
    0.5523,
    0.3854,
    1.0995,
    0.6868,
    0.8366,
    0.704,
    0.6481,
    1.1438,
    1.1345,
    0.6873,
    0.714,
    0.8615,
    1.0099,
    0.9128,
    0.6597,
    1.0033,
    1.1451,
    0.5093,
    0.8667,
    0.3176,
    0.7263,
    0.8744,
    0.9305,
    1.1062,
    0.5864,
    0.7761,
    0.383,
    0.8692,
    1.0822,
    1.0797,
    0.7323,
    0.5558,
    1.1734,
    0.338,
    0.3422,
    0.5202,
    0.7923,
    1.1635,
    1.0173,
    0.4465,
    0.9368,
    0.5635,
    0.5301,
    1.1623,
    0.8049,
    0.8533,
    1.0653,
    0.8663,
    1.1477,
    1.1504,
    0.7187,
    1.1355,
    0.8492,
    0.9118
  ],
  "timed_steps": 62,
  "timed_loop_s": 51.799,
  "own_kernel_ms": {
    "rms_fwd": 284.31,
    "e

In [73]:
!grep -n "entropy\|compute_loss\|cross_entropy\|log_softmax\|accuracy" $(python -c "import soup_cli.trainer.sft as m; print(m.__file__)") | head -40

61:    "entropy",
1113:        # TRL's `compute_loss` reads for its metrics. The schema keeps the
1635:            # returns `logits=None`. Without that flag TRL's entropy metric is not
1636:            # guarded and `entropy_from_logits(None)` raises at step 0, so
2435:        # v0.53.2 #135 — EBFT compute_loss hook (no-op if ebft_variant unset).
2437:        from soup_cli.utils.ebft_gdpo import attach_ebft_compute_loss
2438:        attach_ebft_compute_loss(self.trainer, self.config.training)


In [74]:
%cd /kaggle/working/starter
!python harness/run_task.py --only solution --profile --out runs/prof 2>&1 | tail -60

/kaggle/working
[run_task] warm-up done in 13.69 s
[run_task] solution_1: training ...
[run_task] solution_1: ok, 78.85 s, peak 12506 MiB
[run_task] eval solution ...
[run_task] solution_profile: training ...
[run_task] solution_profile: ok, 122.36 s, peak 12518 MiB

=== result ===
solution  time     78.8 s   peak  12506 MiB   eval loss 2.04965
profiler table: /kaggle/working/starter/runs/prof/solution_profile/profile.json
written: /kaggle/working/starter/runs/prof/result.json


In [75]:
!python - <<'EOF'
import json
d = json.load(open('/kaggle/working/starter/runs/prof/solution_profile/profile.json'))
print(type(d).__name__, list(d)[:10] if isinstance(d, dict) else len(d))
s = json.dumps(d, indent=1)
print(s[:3500])
EOF

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `EOF')
dict ['active_steps', 'total_gpu_us', 'kernels']
{
 "active_steps": 16,
 "total_gpu_us": 12592237.187000105,
 "kernels": [
  {
   "name": "turing_fp16_s1688gemm_fp16_256x128_ldg8_f2f_stages_32x1_nn",
   "time_us": 2047061.5019999999,
   "count": 2180
  },
  {
   "name": "turing_fp16_s1688gemm_fp16_256x128_ldg8_f2f_tn",
   "time_us": 1646507.9469999906,
   "count": 884
  },
  {
   "name": "void at::native::unrolled_elementwise_kernel<at::native::direct_copy_kernel_cuda(at::TensorIteratorBase&)::{lambda()#3}::operator()() const::{lambda()#7}::operator()() const::{lambda(float)#1}, std::array<char*, 2ul>, 4, TrivialOffsetCalculator<1, unsigned int>, TrivialOffsetCalculator<1, unsigned int>, at::native::memory::LoadWithCast<1>, at::native::memory::StoreWithCast<1> >(int, at::native::direct_copy_kernel_cuda(at::TensorIteratorBase&)::{lambda()#3}::operator()() const::{lambda()#7}::operator()() const::

NameError: name 'EOF' is not defined

In [76]:
import json
d=json.load(open('/kaggle/working/starter/runs/prof/solution_profile/profile.json'))
for k in d['kernels'][:25]:
    print(round(k['time_us']/1000), 'ms', k['count'], k['name'][:80])

2047 ms 2180 turing_fp16_s1688gemm_fp16_256x128_ldg8_f2f_stages_32x1_nn
1647 ms 884 turing_fp16_s1688gemm_fp16_256x128_ldg8_f2f_tn
1167 ms 9456 void at::native::unrolled_elementwise_kernel<at::native::direct_copy_kernel_cuda
941 ms 10752 void at::native::vectorized_elementwise_kernel<4, at::native::float16_copy_kerne
861 ms 896 turing_fp16_s1688gemm_fp16_128x128_ldg8_f2f_tn
668 ms 263 void at::native::(anonymous namespace)::cunn_SoftMaxForward<4, float, float, flo
579 ms 6224 void kDequantizeBlockwise<__half, 512, 64, 8, 2>(float*, unsigned char*, float*,
579 ms 448 fmha_cutlassB_f16_aligned_128x64_k128_sm75(PyTorchMemEffAttention::AttentionBack
472 ms 448 example_swiglu_bwd_kernel
430 ms 4848 void at::native::vectorized_elementwise_kernel<4, at::native::CUDAFunctor_add<fl
280 ms 448 example_swiglu_fwd_kernel
267 ms 1175 void at::native::vectorized_elementwise_kernel<4, at::native::BinaryFunctor<floa
259 ms 4000 void at::native::elementwise_kernel<128, 2, at::native::gpu_kernel_impl_no

In [100]:
!grep -n -i -E "quant|4bit|nf4|double" /kaggle/working/starter/task/task.yaml

22:  quantization: 4bit


In [101]:
!grep -rn "dequantize_4bit\|dequantize_blockwise" $(python -c "import bitsandbytes,os; print(os.path.dirname(bitsandbytes.__file__))") --include=*.py | head -30

/usr/local/lib/python3.13/dist-packages/bitsandbytes/nn/parametrize.py:39:        return F.dequantize_4bit(quantized_param, self.quant_state)
/usr/local/lib/python3.13/dist-packages/bitsandbytes/nn/modules.py:958:        output = bnb.functional.dequantize_4bit(output_4bit, output_quant_state)
/usr/local/lib/python3.13/dist-packages/bitsandbytes/nn/modules.py:972:        dequantized_weight = bnb.functional.dequantize_4bit(self.weight.data, self.weight.quant_state)
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/hpu/ops.py:19:@register_kernel("bitsandbytes::dequantize_4bit", "hpu")
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/triton/ops.py:24:def dequantize_blockwise(
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/triton/ops.py:41:def dequantize_blockwise_inplace(
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/triton/ops.py:103:def dequantize_4bit(
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/triton/ops.py:124:

In [83]:
!grep -n -i -B2 -A8 "BitsAndBytesConfig(" $(python -c "import soup_cli.trainer.sft as m; print(m.__file__)") | head -50

In [95]:
!grep -rn -i -A8 "BitsAndBytesConfig(" /usr/local/lib/python3.*/site-packages/soup_cli /opt/conda/lib/python3.*/site-packages/soup_cli 2>/dev/null | head -40

In [99]:
!grep -rn "dequantize_4bit" /usr/local/lib/python3.*/site-packages/bitsandbytes --include=*.py 2>/dev/null | head -20

In [102]:
!echo ok

ok


In [103]:
!find / -name "sft_trainer.py" -path "*trl*" 2>/dev/null

/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py


In [104]:
!grep -n -E "def compute_loss|entropy_from_logits|selective_log_softmax|accuracy|cross_entropy|logits" /usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py | head -50

61:    entropy_from_logits,
66:    selective_log_softmax,
503:    logprobs = selective_log_softmax(outputs.logits, shift_labels)
544:            as a part of the final loss, remember to set the `output_router_logits` config of the model to `True`.
593:        preprocess_logits_for_metrics (`Callable[[torch.Tensor, torch.Tensor], torch.Tensor]`, *optional*):
594:            A function that preprocess the logits right before caching them at each evaluation step. Must take two
595:            tensors, the logits and the labels, and return the logits once processed as desired. The modifications made
622:        preprocess_logits_for_metrics: Callable[[torch.Tensor, torch.Tensor], torch.Tensor] | None = None,
788:        # input. We store the number of these tokens so we can account for them correctly when calculating accuracy.
948:            preprocess_logits_for_metrics=preprocess_logits_for_metrics,
957:        self.aux_loss_enabled = getattr(model.config, "output_router_logits", False)

In [105]:
!sed -n 1236,1262p /usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py

        )

        # Compute entropy
        if not self.args.use_liger_kernel:  # liger doesn't return logits
            with torch.no_grad():
                per_token_entropy = entropy_from_logits(outputs.logits)
                # When using Prompt Tuning, skip the virtual tokens in logits before entropy computation, since they
                # do not correspond to actual input tokens.
                if (
                    self.num_virtual_tokens > 0
                    and model.peft_config[model.active_adapter].peft_type != PeftType.PREFIX_TUNING
                ):
                    per_token_entropy = per_token_entropy[:, self.num_virtual_tokens :]
                if "attention_mask" in inputs:
                    attention_mask = inputs["attention_mask"]
                    entropy = torch.sum(per_token_entropy * attention_mask) / attention_mask.sum()
                elif "position_ids" in inputs:
                    entropy = torch.mean(per_token_entropy)
               

In [106]:
%%writefile /kaggle/working/starter/solution/entropy.py
import torch
import triton
import triton.language as tl


@triton.jit
def entropy_kernel(X, OUT, V, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    base = X + row.to(tl.int64) * V
    m = tl.max(tl.full([BLOCK], -3.0e38, tl.float32), 0)
    for off in range(0, V, BLOCK):
        col = off + tl.arange(0, BLOCK)
        x = tl.load(base + col, mask=col < V, other=-3.0e38).to(tl.float32)
        m = tl.maximum(m, tl.max(x, 0))
    s = tl.sum(tl.zeros([BLOCK], tl.float32), 0)
    t = tl.sum(tl.zeros([BLOCK], tl.float32), 0)
    for off in range(0, V, BLOCK):
        col = off + tl.arange(0, BLOCK)
        ok = col < V
        x = tl.load(base + col, mask=ok, other=-3.0e38).to(tl.float32)
        e = tl.where(ok, tl.exp(x - m), 0.0)
        s += tl.sum(e, 0)
        t += tl.sum(e * tl.where(ok, x, 0.0), 0)
    tl.store(OUT + row, m + tl.log(s) - t / s)


def own_entropy(logits):
    if not logits.is_cuda:
        return _orig(logits)
    shape = logits.shape
    v = shape[-1]
    x = logits.contiguous().view(-1, v)
    out = torch.empty(x.shape[0], device=x.device, dtype=torch.float32)
    entropy_kernel[(x.shape[0],)](x, out, v, 4096)
    return out.view(shape[:-1])


_orig = None


def patch_entropy():
    global _orig
    import trl.trainer.sft_trainer as st
    _orig = st.entropy_from_logits
    st.entropy_from_logits = own_entropy

Writing /kaggle/working/starter/solution/entropy.py


In [107]:
%%writefile /kaggle/working/starter/solution/__init__.py
"""Own Triton kernels."""

from . import swiglu, rmsnorm, entropy


def apply():
    swiglu.patch_qwen2_mlp()
    rmsnorm.patch_rmsnorm()
    entropy.patch_entropy()

Overwriting /kaggle/working/starter/solution/__init__.py


In [108]:
%cd /kaggle/working/starter
!python -c "import torch, solution; from solution import entropy as e; from trl.trainer.sft_trainer import entropy_from_logits as ref; x = torch.randn(2, 8, 151936, device='cuda', dtype=torch.float16) * 3; print((e.own_entropy(x) - ref(x)).abs().max().item())"

/kaggle/working
0.011761188507080078


In [109]:
%cd /kaggle/working/starter
!python -c "import torch, solution; from solution import entropy as e; x = torch.randn(2, 8, 151936, device='cuda', dtype=torch.float16) * 3; p = torch.softmax(x.float(), -1); r = torch.logsumexp(x.float(), -1) - (p * x.float()).sum(-1); print((e.own_entropy(x) - r).abs().max().item())"

/kaggle/working
3.337860107421875e-06


In [110]:
%cd /kaggle/working/starter
!python harness/run_task.py --only solution --out runs/sol2 2>&1 | tail -8
!python -c "import json; d=json.load(open('/kaggle/working/starter/runs/sol2/solution_1/metrics.json')); print(d['own_kernel_ms'], d['timed_loop_s'])"

/kaggle/working
[run_task] warm-up done in 13.64 s
[run_task] solution_1: training ...
[run_task] solution_1: ok, 76.04 s, peak 12506 MiB
[run_task] eval solution ...

=== result ===
solution  time     76.0 s   peak  12506 MiB   eval loss 2.0497
written: /kaggle/working/starter/runs/sol2/result.json
{'rms_fwd': 284.584, 'example_swiglu_fwd_kernel': 1105.644, 'entropy_kernel': 400.135, 'rms_bwd': 384.953, 'example_swiglu_bwd_kernel': 1864.916} 48.226


In [111]:
!grep -rn "def dequantize_4bit\|def _dequantize_4bit_impl\|kdequantize\|dequantize_nf4" /usr/local/lib/python3.13/dist-packages/bitsandbytes --include=*.py | head -20

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/hpu/ops.py:43:    out_dq = torch.ops.hpu.dequantize_nf4(
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/triton/ops.py:103:def dequantize_4bit(
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/triton/ops.py:129:def dequantize_4bit_inplace(
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/triton/kernels_4bit.py:251:def dequantize_nf4_tree(val):
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/triton/kernels_4bit.py:300:    mul_high = dequantize_nf4_tree(higher) * absmax
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/triton/kernels_4bit.py:301:    mul_low = dequantize_nf4_tree(lower) * absmax
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/triton/kernels_4bit.py:456:def dequantize_4bit_impl(
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/triton/kernels_4bit.py:481:def dequantize_4bit_impl_passing_code(
/usr/local/lib/python3.13/dis

In [112]:
!sed -n 454,490p /usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py

def _dequantize_4bit_impl(
    A: torch.Tensor,
    absmax: torch.Tensor,
    blocksize: int,
    quant_type: str,
    dtype: torch.dtype,
    out: torch.Tensor,
) -> None:
    A = A.contiguous()

    if dtype == torch.bfloat16:
        if quant_type == "fp4":
            fn = lib.cdequantize_blockwise_bf16_fp4
        else:
            fn = lib.cdequantize_blockwise_bf16_nf4
    elif dtype == torch.float16:
        if quant_type == "fp4":
            fn = lib.cdequantize_blockwise_fp16_fp4
        else:
            fn = lib.cdequantize_blockwise_fp16_nf4
    elif dtype == torch.float32:
        if quant_type == "fp4":
            fn = lib.cdequantize_blockwise_fp32_fp4
        else:
            fn = lib.cdequantize_blockwise_fp32_nf4
    else:
        raise ValueError(f"Blockwise 4bit dequantization only supports 16/32-bit floats, but got {dtype}")

    with _cuda_device_of(A):
        fn(
            None,
            A.data_ptr(),
            absmax.data_ptr(),
            out.data_

In [113]:
!sed -n 992,1050p /usr/local/lib/python3.13/dist-packages/bitsandbytes/functional.py

def dequantize_4bit(
    A: torch.Tensor,
    quant_state: Optional[QuantState] = None,
    absmax: Optional[torch.Tensor] = None,
    out: Optional[torch.Tensor] = None,
    blocksize: Optional[int] = None,
    quant_type="fp4",
) -> torch.Tensor:
    """Dequantizes a packed 4-bit quantized tensor.

    The input tensor is dequantized by dividing it into blocks of `blocksize` values.
    The absolute maximum value within these blocks is used for scaling
    the non-linear dequantization.

    Args:
        A (`torch.Tensor`): The quantized input tensor.
        quant_state ([`QuantState`], *optional*):
            The quantization state as returned by [`quantize_4bit`].
            Required if `absmax` is not provided.
        absmax (`torch.Tensor`, *optional*):
            A tensor containing the scaling values.
            Required if `quant_state` is not provided and ignored otherwise.
        out (`torch.Tensor`, *optional*): A tensor to use to store the result.
        blocksize

In [114]:
!grep -rn -i "double_quant\|bnb_4bit" /usr/local/lib/python3.13/dist-packages/soup_cli --include=*.py | head -10

/usr/local/lib/python3.13/dist-packages/soup_cli/commands/train.py:1302:    original_quant_storage = cfg.training.bnb_4bit_quant_storage
/usr/local/lib/python3.13/dist-packages/soup_cli/commands/train.py:1311:            f"[green]FSDP QLoRA:[/] {action} bnb_4bit_quant_storage="
/usr/local/lib/python3.13/dist-packages/soup_cli/commands/train.py:1312:            f"{resolved_training.bnb_4bit_quant_storage} to match compute dtype"
/usr/local/lib/python3.13/dist-packages/soup_cli/commands/train.py:1322:        bnb_4bit_quant_storage=cfg.training.bnb_4bit_quant_storage,
/usr/local/lib/python3.13/dist-packages/soup_cli/commands/merge.py:58:    double_quant: bool = typer.Option(
/usr/local/lib/python3.13/dist-packages/soup_cli/commands/merge.py:251:                    double_quant=double_quant,
/usr/local/lib/python3.13/dist-packages/soup_cli/trainer/stream_setup.py:441:    double_quant: bool = True,
/usr/local/lib/python3.13/dist-packages/soup_cli/trainer/stream_setup.py:514:                

In [115]:
%%bash
cd /kaggle/working/starter
cat > solution/nf4.py <<'PYEOF'
import torch
import triton
import triton.language as tl

_NF4 = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
        -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
        0.07958029955625534, 0.16093020141124725, 0.24611230194568634,
        0.33791524171829224, 0.44070982933044434, 0.5626170039176392,
        0.7229568362236023, 1.0]
_lut = {}
STATS = {"calls": 0}


@triton.jit
def nf4_dequant_kernel(A, ABSMAX, LUT, OUT, NBYTES,
                       HALF_BLOCK: tl.constexpr, BLOCK: tl.constexpr):
    pid = tl.program_id(0).to

bash: line 19: warning: here-document at line 2 delimited by end-of-file (wanted `PYEOF')


In [116]:
%%writefile /kaggle/working/starter/solution/nf4.py
import torch
import triton
import triton.language as tl

_NF4 = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
        -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
        0.07958029955625534, 0.16093020141124725, 0.24611230194568634,
        0.33791524171829224, 0.44070982933044434, 0.5626170039176392,
        0.7229568362236023, 1.0]
_lut = {}
STATS = {"calls": 0}


@triton.jit
def nf4_dequant_kernel(A, ABSMAX, LUT, OUT, NBYTES,
                       HALF_BLOCK: tl.constexpr, BLOCK: tl.constexpr):
    pid = tl.program_id(0).to(tl.int64)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < NBYTES
    b = tl.load(A + offs, mask=mask, other=0).to(tl.int32)
    am = tl.load(ABSMAX + offs // HALF_BLOCK, mask=mask, other=0.0)
    hi = tl.load(LUT + (b >> 4))
    lo = tl.load(LUT + (b & 15))
    res = tl.interleave((hi * am).to(tl.float16), (lo * am).to(tl.float16))
    o = pid * (2 * BLOCK) + tl.arange(0, 2 * BLOCK)
    tl.store(OUT + o, res, mask=o < 2 * NBYTES)


def _lut_for(device):
    t = _lut.get(device)
    if t is None:
        t = torch.tensor(_NF4, dtype=torch.float32, device=device)
        _lut[device] = t
    return t


def dequant_nf4(A, absmax, blocksize, out):
    nbytes = A.numel()
    block = 1024
    grid = (triton.cdiv(nbytes, block),)
    nf4_dequ

Overwriting /kaggle/working/starter/solution/nf4.py


In [117]:
%%writefile /kaggle/working/starter/solution/__init__.py
"""Own Triton kernels."""

from . import swiglu, rmsnorm, entropy, nf4


def apply():
    swiglu.patch_qwen2_mlp()
    rmsnorm.patch_rmsnorm()
    entropy.patch_entropy()
    nf4.patch_nf4()

Overwriting /kaggle/working/starter/solution/__init__.py


In [119]:
%cd /kaggle/working/starter
!python -c "import torch, bitsandbytes as bnb; from solution import nf4; w = torch.randn(8960, 1536, device='cuda', dtype=torch.float16); data = [bnb.functional.quantize_4bit(w, blocksize=64, quant_type='nf4', compress_statistics=c) for c in (False, True)]; refs = [bnb.functional.dequantize_4bit(q, s) for q, s in data]; nf4.patch_nf4(); [print('equal:', torch.equal(bnb.functional.dequantize_4bit(q, s), r), 'calls:', nf4.STATS['calls']) for (q, s), r in zip(data, refs)]"

/kaggle/working
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import torch, bitsandbytes as bnb; from solution import nf4; w = torch.randn(8960, 1536, device='cuda', dtype=torch.float16); data = [bnb.functional.quantize_4bit(w, blocksize=64, quant_type='nf4', compress_statistics=c) for c in (False, True)]; refs = [bnb.functional.dequantize_4bit(q, s) for q, s in data]; nf4.patch_nf4(); [print('equal:', torch.equal(bnb.functional.dequantize_4bit(q, s), r), 'calls:', nf4.STATS['calls']) for (q, s), r in zip(data, refs)]
                                                                                                                                                                                                                                                                                                                          ^^^^^^^^^^^^^
AttributeError: module 'solution.nf4' has no attribute 'patch_nf4'


In [120]:
!grep -n "def " /kaggle/working/starter/solution/nf4.py

15:def nf4_dequant_kernel(A, ABSMAX, LUT, OUT, NBYTES,
29:def _lut_for(device):
37:def dequant_nf4(A, absmax, blocksize, out):


In [121]:
%%writefile /kaggle/working/starter/solution/nf4.py
import torch
import triton
import triton.language as tl

_NF4 = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176392, 0.7229568362236023, 1.0]
_lut = {}
STATS = {"calls": 0}


@triton.jit
def nf4_dequant_kernel(A, ABSMAX, LUT, OUT, NBYTES,
                       HALF_BLOCK: tl.constexpr, BLOCK: tl.constexpr):
    pid = tl.program_id(0).to(tl.int64)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < NBYTES
    b = tl.load(A + offs, mask=mask, other=0).to(tl.int32)
    am = tl.load(ABSMAX + offs // HALF_BLOCK, mask=mask, other=0.0)
    hi = tl.load(LUT + (b >> 4))
    lo = tl.load(LUT + (b & 15))
    res = tl.interleave((hi * am).to(tl.float16), (lo * am).to(tl.float16))
    o = pid * (2 * BLOCK) + tl.arange(0, 2 * BLOCK)
    tl.store(OUT + o, res, mask=o < 2 * NBYTES)


def _lut_for(device):
    t = _lut.get(device)
    if t is None:
        t = torch.tensor(_NF4, dtype=torch.float32, device=device)
        _lut[device] = t
    return t


def dequant_nf4(A, absmax, blocksize, out):
    nbytes = A.numel()
    grid = (triton.cdiv(nbytes, 1024),)
    nf4_dequant_kernel[grid](A, absmax, _lut_for(A.device), out,
                             nbytes, blocksize // 2, 1024)

Overwriting /kaggle/working/starter/solution/nf4.py


In [122]:
%%writefile -a /kaggle/working/starter/solution/nf4.py


def patch_nf4():
    from bitsandbytes.backends.cuda import ops
    orig = ops._dequantize_4bit_impl

    def impl(A, absmax, blocksize, quant_type, dtype, out):
        if (quant_type == "nf4" and dtype == torch.float16
                and A.is_cuda and A.dtype == torch.uint8
                and absmax.dtype == torch.float32
                and A.is_contiguous() and absmax.is_contiguous()
                and out.is_contiguous()
                and A.numel() * 2 == out.numel()):
            STATS["calls"] += 1
            dequant_nf4(A, absmax, blocksize, out)
        else:
            orig(A, absmax, blocksize, quant_type, dtype, out)

    ops._dequantize_4bit_impl = impl

Appending to /kaggle/working/starter/solution/nf4.py


In [123]:
!grep -n "def " /kaggle/working/starter/solution/nf4.py

11:def nf4_dequant_kernel(A, ABSMAX, LUT, OUT, NBYTES,
25:def _lut_for(device):
33:def dequant_nf4(A, absmax, blocksize, out):
40:def patch_nf4():
44:    def impl(A, absmax, blocksize, quant_type, dtype, out):


In [124]:
%cd /kaggle/working/starter
!python -c "import torch, bitsandbytes as bnb; from solution import nf4; w = torch.randn(8960, 1536, device='cuda', dtype=torch.float16); data = [bnb.functional.quantize_4bit(w, blocksize=64, quant_type='nf4', compress_statistics=c) for c in (False, True)]; refs = [bnb.functional.dequantize_4bit(q, s) for q, s in data]; nf4.patch_nf4(); [print('equal:', torch.equal(bnb.functional.dequantize_4bit(q, s), r), 'calls:', nf4.STATS['calls']) for (q, s), r in zip(data, refs)]"

/kaggle/working
equal: True calls: 1
equal: True calls: 2


In [125]:
%cd /kaggle/working/starter
!python harness/run_task.py --only solution --out runs/sol3 2>&1 | tail -6
!python -c "import json; d=json.load(open('/kaggle/working/starter/runs/sol3/solution_1/metrics.json')); k=d['own_kernel_ms']; own=sum(v for n,v in k.items() if not n.startswith('example')); print(k); print('share %:', round(100*own/1000/d['timed_loop_s'],2))"

/kaggle/working
[run_task] solution_1: ok, 76.59 s, peak 12506 MiB
[run_task] eval solution ...

=== result ===
solution  time     76.6 s   peak  12506 MiB   eval loss 2.0497
written: /kaggle/working/starter/runs/sol3/result.json
{'rms_fwd': 286.676, 'nf4_dequant_kernel': 2044.004, 'example_swiglu_fwd_kernel': 1106.006, 'entropy_kernel': 400.105, 'rms_bwd': 386.052, 'example_swiglu_bwd_kernel': 1865.115}
share %: 6.45


In [127]:
%cd /kaggle/working/starter
!python harness/run_task.py 2>&1 | tail -15

/kaggle/working
[run_task] warm-up done in 13.89 s
[run_task] base_1: training ...
[run_task] base_1: ok, 79.86 s, peak 14468 MiB
[run_task] solution_1: training ...
[run_task] solution_1: ok, 78.2 s, peak 12506 MiB
[run_task] eval base ...
[run_task] eval solution ...

=== result ===
base      time     79.9 s   peak  14468 MiB   eval loss 2.04967
solution  time     78.2 s   peak  12506 MiB   eval loss 2.04962
speedup 1.021x   memory base/solution 1.157x   eval loss vs base -0.002 %
written: /kaggle/working/starter/runs/result.json


In [128]:
%cd /kaggle/working/starter
!python harness/check_rules.py 2>&1 | tail -30

/kaggle/working
PASS  1. every base and solution run finished                                            1 base and 1 solution runs of 1 each
PASS  2. no ready-made kernel libraries                                                  loaded: none; imported in solution/: none; packages changed during the run: none
PASS  3. own @triton.jit kernels in solution/ run in training                            entropy_kernel (solution/entropy.py), nf4_dequant_kernel (solution/nf4.py), rms_bwd (solution/rmsnorm.py), rms_fwd (solution/rmsnorm.py); the example's kernels do not count, under any name: example_swiglu_fwd_kernel, example_swiglu_bwd_kernel
PASS  4. own kernels together take >= 5% of the training time                            6.3% of the training time (lowest of 1 runs, steps 3+)
PASS  5. same task and Soup outputs (steps, adapter, settings, tokens, metrics, rewind)  identical within tolerance
PASS  6. loss curve follows the base                                                     solutio

In [129]:
!grep -n -i -E "submit|submission|deadline|score|leaderboard" /kaggle/working/starter/README.md

5:October 30 12:00 – October 31 23:59 (Astana time, UTC+5; the deadline is October 31 18:59 UTC).
124:| `harness/score.py` | ratios and points | no |
127:| `templates/` | Kaggle notebook, `SUBMISSION.md` | copy and fill |
133:  compiler caches. Base and solution run in the same session; the score uses the ratio.
208:memory points. `harness/score.py` prints your ratios.
214:## Submit
216:Everything is submitted on the hackathon platform: your repository, the video (up to 3 minutes)
219:`templates/SUBMISSION.md` and put it in the root of your repository: links to the final commit,
221:presentation. The Kaggle link must be in both the README and `SUBMISSION.md`. By the deadline the


In [130]:
!sed -n 205,226p /kaggle/working/starter/README.md
!cat /kaggle/working/starter/templates/SUBMISSION.md


A gain under 5% counts as a tie (ratio 1.0). "Best" is the best admitted team. A team that fails a
check, or whose code or README the jury does not accept, is not admitted and gets no speed or
memory points. `harness/score.py` prints your ratios.

The jury gives the README points in three parts, up to 5 each: what you did, how to run it,
results and limits. If two teams have the same total, the one with the better memory result ranks
higher.

## Submit

Everything is submitted on the hackathon platform: your repository, the video (up to 3 minutes)
and the presentation. Rewrite this README for your solution (what you did, how to run it,
results, limits) and put the link to your public Kaggle notebook in it. Fill
`templates/SUBMISSION.md` and put it in the root of your repository: links to the final commit,
the public Kaggle notebook (a saved version with `COMMIT` set to that commit), the video and the
presentation. The Kaggle link must be in both the README and `SUBMISSION.md`. By the 

In [131]:
import shutil
shutil.make_archive('/kaggle/working/solution_backup', 'zip', '/kaggle/working/starter', 'solution')

'/kaggle/working/solution_backup.zip'

In [132]:
!cd /kaggle/working/starter/solution && for f in __init__.py rmsnorm.py entropy.py nf4.py; do echo "=== $f ==="; cat $f; done

=== /kaggle/working/starter/solution/test_kernels.py ===
"""Correctness of your kernels against plain PyTorch, forward and backward.

Run on a GPU: `python -m pytest solution -q`. Add a test like this for every kernel you write.
"""

import pytest
import torch

pytestmark = pytest.mark.skipif(not torch.cuda.is_available(), reason="needs a CUDA GPU")

TOLERANCE = {torch.float16: 2e-3, torch.float32: 1e-4}


@pytest.mark.parametrize("dtype", [torch.float16, torch.float32])
@pytest.mark.parametrize("shape", [(4, 512, 8960), (3, 77, 1001)])
def test_swiglu_matches_pytorch(dtype, shape):
    from solution.swiglu import swiglu

    torch.manual_seed(0)
    gate = torch.randn(shape, device="cuda", dtype=dtype, requires_grad=True)
    up = torch.randn(shape, device="cuda", dtype=dtype, requires_grad=True)
    grad = torch.randn(shape, device="cuda", dtype=dtype)

    ref = torch.nn.functional.silu(gate.float()) * up.float()
    ref.backward(grad.float())
    ref_dgate, ref_dup = gate.grad.floa

In [133]:
import os, zipfile
root = '/kaggle/working/starter'
skip = {'data', 'runs', '.git', '__pycache__', '.pytest_cache'}
n = 0
with zipfile.ZipFile('/kaggle/working/kit.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for d, dirs, files in os.walk(root):
        dirs[:] = [x for x in dirs if x not in skip]
        for f in files:
            p = os.path.join(d, f)
            z.write(p, os.path.relpath(p, root))
            n += 1
print(n, 'файлов,', os.path.getsize('/kaggle/working/kit.zip') // 1024, 'KB')

144 файлов, 45892 KB


In [134]:
!ls -la /kaggle/working/*.zip

-rw-r--r-- 1 root root 46993949 Oct 31 03:38 /kaggle/working/kit.zip
-rw-r--r-- 1 root root    17413 Oct 31 03:26 /kaggle/working/solution_backup.zip
